In [35]:
import pandas as pd

df = pd.read_csv(
    "IMDB Dataset.csv",
    engine="python",
    on_bad_lines="skip"
)

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (50000, 2)
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [36]:
# Number of words
df["word_count"] = df["review"].str.split().str.len()

# Longest review by words
longest_review = df.loc[df["word_count"].idxmax()]

print("Longest review:")
print(longest_review["review"])

print("\nTotal words in longest review:", longest_review["word_count"])

Longest review:
Match 1: Tag Team Table Match Bubba Ray and Spike Dudley vs Eddie Guerrero and Chris Benoit Bubba Ray and Spike Dudley started things off with a Tag Team Table Match against Eddie Guerrero and Chris Benoit. According to the rules of the match, both opponents have to go through tables in order to get the win. Benoit and Guerrero heated up early on by taking turns hammering first Spike and then Bubba Ray. A German suplex by Benoit to Bubba took the wind out of the Dudley brother. Spike tried to help his brother, but the referee restrained him while Benoit and Guerrero ganged up on him in the corner. With Benoit stomping away on Bubba, Guerrero set up a table outside. Spike dashed into the ring and somersaulted over the top rope onto Guerrero on the outside! After recovering and taking care of Spike, Guerrero slipped a table into the ring and helped the Wolverine set it up. The tandem then set up for a double superplex from the middle rope which would have put Bubba throug

In [37]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

df["token_count"] = df["review"].apply(
    lambda x: len(enc.encode(x))
)

print("Maximum tokens:", df["token_count"].max())

longest_token_review = df.loc[df["token_count"].idxmax()]

print("\nReview with maximum tokens:")
print(longest_token_review["review"])

print("\nToken count:", longest_token_review["token_count"])
print("Word count:", longest_token_review["word_count"])

Maximum tokens: 3112

Review with maximum tokens:
Match 1: Tag Team Table Match Bubba Ray and Spike Dudley vs Eddie Guerrero and Chris Benoit Bubba Ray and Spike Dudley started things off with a Tag Team Table Match against Eddie Guerrero and Chris Benoit. According to the rules of the match, both opponents have to go through tables in order to get the win. Benoit and Guerrero heated up early on by taking turns hammering first Spike and then Bubba Ray. A German suplex by Benoit to Bubba took the wind out of the Dudley brother. Spike tried to help his brother, but the referee restrained him while Benoit and Guerrero ganged up on him in the corner. With Benoit stomping away on Bubba, Guerrero set up a table outside. Spike dashed into the ring and somersaulted over the top rope onto Guerrero on the outside! After recovering and taking care of Spike, Guerrero slipped a table into the ring and helped the Wolverine set it up. The tandem then set up for a double superplex from the middle rope

In [38]:
print("Original dataset shape:", df.shape)

# Take 3000 reviews for lab-speed training
df = df.sample(
    n=3000,
    random_state=42
).reset_index(drop=True)

print("Sampled dataset shape:", df.shape)

# Display first few rows
print(df.head())

Original dataset shape: (50000, 4)
Sampled dataset shape: (3000, 4)
                                              review sentiment  word_count  \
0  I really liked this Summerslam due to the look...  positive         201   
1  Not many television shows appeal to quite as m...  positive         354   
2  The film quickly gets to a major chase scene w...  negative         119   
3  Jane Austen would definitely approve of this o...  positive          99   
4  Expectations were somewhat high for me when I ...  negative         332   

   token_count  
0          260  
1          437  
2          139  
3          143  
4          423  


In [39]:
# Convert sentiment to numerical labels
df["label"] = df["sentiment"].map({
    "positive": 1,
    "negative": 0
})

print(df[["sentiment", "label"]].head(10))

  sentiment  label
0  positive      1
1  positive      1
2  negative      0
3  positive      1
4  negative      0
5  positive      1
6  positive      1
7  positive      1
8  negative      0
9  negative      0


In [40]:
print("\nClass distribution:")
print(df["label"].value_counts())


Class distribution:
label
1    1509
0    1491
Name: count, dtype: int64


In [41]:
from sklearn.model_selection import train_test_split

X = df["review"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 2400
Testing samples: 600


In [42]:

enc = tiktoken.get_encoding("cl100k_base")

In [43]:
sample_review = X_train.iloc[0]

tokens = enc.encode(sample_review)

print("Review:")
print(sample_review[:500])

print("\nFirst 20 token IDs:")
print(tokens[:20])

print("\nNumber of tokens:", len(tokens))

Review:
Well...I like this movie first of all because it's very well thought of... and well..because the um...director and others chose an extremely great actor to play Mike....and also my last reason because ( my opinion) Elijah Wood is so so hot!!!

First 20 token IDs:
[11649, 1131, 40, 1093, 420, 5818, 1176, 315, 682, 1606, 433, 596, 1633, 1664, 3463, 315, 1131, 323, 1664, 497]

Number of tokens: 53


In [44]:
MAX_LEN = 256
PAD_ID = enc.n_vocab

def encode_and_pad(text):
    tokens = enc.encode(text)

    # Truncate
    tokens = tokens[:MAX_LEN]

    # Actual sequence length before padding
    length = len(tokens)

    # Pad
    if len(tokens) < MAX_LEN:
        tokens += [PAD_ID] * (MAX_LEN - len(tokens))

    return tokens, length

In [45]:
tokens, length = encode_and_pad(X_train.iloc[0])

print("Sequence length before padding:", length)
print("Final sequence length:", len(tokens))
print("First 20 tokens:", tokens[:20])

Sequence length before padding: 53
Final sequence length: 256
First 20 tokens: [11649, 1131, 40, 1093, 420, 5818, 1176, 315, 682, 1606, 433, 596, 1633, 1664, 3463, 315, 1131, 323, 1664, 497]


In [46]:
X_train_encoded = []
X_train_lengths = []

for review in X_train:
    tokens, length = encode_and_pad(review)
    X_train_encoded.append(tokens)
    X_train_lengths.append(length)


X_test_encoded = []
X_test_lengths = []

for review in X_test:
    tokens, length = encode_and_pad(review)
    X_test_encoded.append(tokens)
    X_test_lengths.append(length)

In [47]:
import torch

X_train_tensor = torch.tensor(
    X_train_encoded,
    dtype=torch.long
)

X_test_tensor = torch.tensor(
    X_test_encoded,
    dtype=torch.long
)

train_lengths = torch.tensor(
    X_train_lengths,
    dtype=torch.long
)

test_lengths = torch.tensor(
    X_test_lengths,
    dtype=torch.long
)

y_train_tensor = torch.tensor(
    y_train.values,
    dtype=torch.float32
)

y_test_tensor = torch.tensor(
    y_test.values,
    dtype=torch.float32
)

In [48]:
print("X_train:", X_train_tensor.shape)
print("X_test :", X_test_tensor.shape)

print("y_train:", y_train_tensor.shape)
print("y_test :", y_test_tensor.shape)

X_train: torch.Size([2400, 256])
X_test : torch.Size([600, 256])
y_train: torch.Size([2400])
y_test : torch.Size([600])


In [49]:
vocab_size = enc.n_vocab + 1

model = RNNClassifier(
    vocab_size=vocab_size,
    embedding_dim=64,
    hidden_dim=64
)

print(model)

RNNClassifier(
  (embedding): Embedding(100278, 64, padding_idx=100277)
  (rnn): RNN(64, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)


In [50]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [51]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(
    X_train_tensor,
    train_lengths,
    y_train_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    test_lengths,
    y_test_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [52]:
EPOCHS = 5

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for batch_idx, (inputs, lengths, labels) in enumerate(train_loader):

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs, lengths)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )
        # Update weights
        optimizer.step()

        total_loss += loss.item()

        # Print periodically
        if (batch_idx + 1) % 20 == 0:
            print(
                f"Epoch [{epoch+1}/{EPOCHS}], "
                f"Batch [{batch_idx+1}/{len(train_loader)}], "
                f"Loss: {loss.item():.4f}"
            )

    average_loss = total_loss / len(train_loader)

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Average Loss: {average_loss:.4f}"
    )

Epoch [1/5], Batch [20/75], Loss: 0.6938
Epoch [1/5], Batch [40/75], Loss: 0.6906
Epoch [1/5], Batch [60/75], Loss: 0.7255
Epoch [1/5] Average Loss: 0.6952
Epoch [2/5], Batch [20/75], Loss: 0.6427
Epoch [2/5], Batch [40/75], Loss: 0.6945
Epoch [2/5], Batch [60/75], Loss: 0.6620
Epoch [2/5] Average Loss: 0.6742
Epoch [3/5], Batch [20/75], Loss: 0.6377
Epoch [3/5], Batch [40/75], Loss: 0.6861
Epoch [3/5], Batch [60/75], Loss: 0.6852
Epoch [3/5] Average Loss: 0.6417
Epoch [4/5], Batch [20/75], Loss: 0.6038
Epoch [4/5], Batch [40/75], Loss: 0.5081
Epoch [4/5], Batch [60/75], Loss: 0.7227
Epoch [4/5] Average Loss: 0.5954
Epoch [5/5], Batch [20/75], Loss: 0.5500
Epoch [5/5], Batch [40/75], Loss: 0.5018
Epoch [5/5], Batch [60/75], Loss: 0.6146
Epoch [5/5] Average Loss: 0.5241


In [53]:
# Check accuracy on training data

model.eval()

correct = 0
total = 0

with torch.no_grad():

    for inputs, lengths, labels in train_loader:

        outputs = model(inputs, lengths)

        probabilities = torch.sigmoid(outputs)

        predictions = (probabilities >= 0.5).float()

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

train_accuracy = correct / total

print(f"Training Accuracy: {train_accuracy * 100:.2f}%")

Training Accuracy: 80.00%


In [54]:
# Check accuracy on test data

model.eval()

correct = 0
total = 0

with torch.no_grad():

    for inputs, lengths, labels in test_loader:

        outputs = model(inputs, lengths)

        probabilities = torch.sigmoid(outputs)

        predictions = (probabilities >= 0.5).float()

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

test_accuracy = correct / total

print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

Test Accuracy: 56.00%


In [69]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Evaluate RNN
model.eval()

rnn_predictions = []
rnn_actual = []

with torch.no_grad():

    for inputs, lengths, labels in test_loader:

        outputs = model(inputs, lengths)

        probabilities = torch.sigmoid(outputs)

        predictions = (probabilities >= 0.5).float()

        rnn_predictions.extend(predictions.cpu().numpy())
        rnn_actual.extend(labels.cpu().numpy())


# Calculate metrics
rnn_precision = precision_score(rnn_actual, rnn_predictions)
rnn_recall = recall_score(rnn_actual, rnn_predictions)
rnn_f1 = f1_score(rnn_actual, rnn_predictions)

print("RNN Performance:")
print(f"Precision: {rnn_precision:.4f}")
print(f"Recall:    {rnn_recall:.4f}")
print(f"F1 Score:  {rnn_f1:.4f}")

RNN Performance:
Precision: 0.5546
Recall:    0.6391
F1 Score:  0.5938


In [55]:
# Check class distribution

model.eval()

predictions = []
actual = []

with torch.no_grad():

    for inputs, lengths, labels in test_loader:

        outputs = model(inputs, lengths)

        probabilities = torch.sigmoid(outputs)

        preds = (probabilities >= 0.5).float()

        predictions.extend(preds.cpu().numpy())
        actual.extend(labels.cpu().numpy())


import numpy as np

predictions = np.array(predictions)
actual = np.array(actual)

print("Actual class distribution:")
print("Negative (0):", np.sum(actual == 0))
print("Positive (1):", np.sum(actual == 1))

print("\nPredicted class distribution:")
print("Negative (0):", np.sum(predictions == 0))
print("Positive (1):", np.sum(predictions == 1))

Actual class distribution:
Negative (0): 298
Positive (1): 302

Predicted class distribution:
Negative (0): 252
Positive (1): 348


In [56]:
print("Training label distribution:")
print(y_train.value_counts())

print("\nTest label distribution:")
print(y_test.value_counts())

Training label distribution:
label
1    1207
0    1193
Name: count, dtype: int64

Test label distribution:
label
1    302
0    298
Name: count, dtype: int64


In [57]:
print("First 10 y_train labels:")
print(y_train.values[:10])

print("\nFirst 10 y_train_tensor labels:")
print(y_train_tensor[:10])

print("\nFirst 10 y_test labels:")
print(y_test.values[:10])

print("\nFirst 10 y_test_tensor labels:")
print(y_test_tensor[:10])

First 10 y_train labels:
[1 0 0 1 1 1 1 0 1 0]

First 10 y_train_tensor labels:
tensor([1., 0., 0., 1., 1., 1., 1., 0., 1., 0.])

First 10 y_test labels:
[1 1 0 1 0 1 1 1 0 1]

First 10 y_test_tensor labels:
tensor([1., 1., 0., 1., 0., 1., 1., 1., 0., 1.])


In [58]:
test_reviews = [
    "This movie was absolutely fantastic. I loved every minute of it.",
    "This was one of the worst movies I have ever watched. Completely boring.",
    "Amazing acting, wonderful story, and an excellent ending.",
    "Terrible movie. Poor acting and a very disappointing story."
]

def predict_sentiment(review):

    model.eval()

    tokens, length = encode_and_pad(review)

    input_tensor = torch.tensor(
        [tokens],
        dtype=torch.long
    )

    length_tensor = torch.tensor(
        [length],
        dtype=torch.long
    )

    with torch.no_grad():

        output = model(
            input_tensor,
            length_tensor
        )

        probability = torch.sigmoid(output).item()

    if probability >= 0.5:
        sentiment = "Positive"
    else:
        sentiment = "Negative"

    return sentiment, probability


for review in test_reviews:

    sentiment, probability = predict_sentiment(review)

    print("\nReview:", review)
    print("Prediction:", sentiment)
    print(f"Positive probability: {probability:.4f}")


Review: This movie was absolutely fantastic. I loved every minute of it.
Prediction: Positive
Positive probability: 0.7015

Review: This was one of the worst movies I have ever watched. Completely boring.
Prediction: Negative
Positive probability: 0.1069

Review: Amazing acting, wonderful story, and an excellent ending.
Prediction: Positive
Positive probability: 0.8243

Review: Terrible movie. Poor acting and a very disappointing story.
Prediction: Positive
Positive probability: 0.7532


# Activity 2 — Text Classification with LSTM

In [59]:
import torch
import torch.nn as nn

class LSTMClassifier(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=64,
        hidden_dim=64
    ):
        super().__init__()

        # Embedding layer
        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=PAD_ID
        )

        # LSTM layer
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        # Linear output layer
        self.fc = nn.Linear(
            hidden_dim,
            1
        )

    def forward(self, x, lengths):

        # Token IDs → embeddings
        embedded = self.embedding(x)

        # Ignore padded tokens
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        # LSTM
        packed_output, (hidden, cell) = self.lstm(packed)

        # Last hidden state
        last_hidden = hidden[-1]

        # Linear classification layer
        output = self.fc(last_hidden)

        return output.squeeze(1)

In [60]:
vocab_size = enc.n_vocab + 1

lstm_model = LSTMClassifier(
    vocab_size=vocab_size,
    embedding_dim=64,
    hidden_dim=64
)

print(lstm_model)

LSTMClassifier(
  (embedding): Embedding(100278, 64, padding_idx=100277)
  (lstm): LSTM(64, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)


In [61]:
criterion_lstm = nn.BCEWithLogitsLoss()

optimizer_lstm = torch.optim.Adam(
    lstm_model.parameters(),
    lr=0.001
)

In [62]:
EPOCHS = 5

for epoch in range(EPOCHS):

    lstm_model.train()

    total_loss = 0
    correct = 0
    total = 0

    for batch_idx, (inputs, lengths, labels) in enumerate(train_loader):

        optimizer_lstm.zero_grad()

        # Forward pass
        outputs = lstm_model(inputs, lengths)

        # Calculate loss
        loss = criterion_lstm(outputs, labels)

        # Backpropagation
        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(
            lstm_model.parameters(),
            max_norm=1.0
        )

        # Update parameters
        optimizer_lstm.step()

        total_loss += loss.item()

        # Training accuracy
        probabilities = torch.sigmoid(outputs)

        predictions = (probabilities >= 0.5).float()

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        # Print loss periodically
        if (batch_idx + 1) % 20 == 0:
            print(
                f"Epoch [{epoch+1}/{EPOCHS}], "
                f"Batch [{batch_idx+1}/{len(train_loader)}], "
                f"Loss: {loss.item():.4f}"
            )

    average_loss = total_loss / len(train_loader)
    train_accuracy = correct / total

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Average Loss: {average_loss:.4f} "
        f"Training Accuracy: {train_accuracy * 100:.2f}%"
    )

Epoch [1/5], Batch [20/75], Loss: 0.6818
Epoch [1/5], Batch [40/75], Loss: 0.6950
Epoch [1/5], Batch [60/75], Loss: 0.6869
Epoch [1/5] Average Loss: 0.6912 Training Accuracy: 51.96%
Epoch [2/5], Batch [20/75], Loss: 0.6620
Epoch [2/5], Batch [40/75], Loss: 0.6573
Epoch [2/5], Batch [60/75], Loss: 0.7202
Epoch [2/5] Average Loss: 0.6707 Training Accuracy: 58.92%
Epoch [3/5], Batch [20/75], Loss: 0.5582
Epoch [3/5], Batch [40/75], Loss: 0.6798
Epoch [3/5], Batch [60/75], Loss: 0.5572
Epoch [3/5] Average Loss: 0.6307 Training Accuracy: 64.25%
Epoch [4/5], Batch [20/75], Loss: 0.6738
Epoch [4/5], Batch [40/75], Loss: 0.5716
Epoch [4/5], Batch [60/75], Loss: 0.5513
Epoch [4/5] Average Loss: 0.5555 Training Accuracy: 72.25%
Epoch [5/5], Batch [20/75], Loss: 0.5830
Epoch [5/5], Batch [40/75], Loss: 0.3500
Epoch [5/5], Batch [60/75], Loss: 0.3864
Epoch [5/5] Average Loss: 0.4450 Training Accuracy: 80.25%


In [64]:
lstm_model.eval()

correct = 0
total = 0

with torch.no_grad():

    for inputs, lengths, labels in test_loader:

        outputs = lstm_model(inputs, lengths)

        probabilities = torch.sigmoid(outputs)

        predictions = (probabilities >= 0.5).float()

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

lstm_test_accuracy = correct / total

print(
    f"LSTM Test Accuracy: "
    f"{lstm_test_accuracy * 100:.2f}%"
)

LSTM Test Accuracy: 59.83%


In [70]:
# Evaluate LSTM
lstm_model.eval()

lstm_predictions = []
lstm_actual = []

with torch.no_grad():

    for inputs, lengths, labels in test_loader:

        outputs = lstm_model(inputs, lengths)

        probabilities = torch.sigmoid(outputs)

        predictions = (probabilities >= 0.5).float()

        lstm_predictions.extend(predictions.cpu().numpy())
        lstm_actual.extend(labels.cpu().numpy())


# Calculate metrics
lstm_precision = precision_score(lstm_actual, lstm_predictions)
lstm_recall = recall_score(lstm_actual, lstm_predictions)
lstm_f1 = f1_score(lstm_actual, lstm_predictions)

print("LSTM Performance:")
print(f"Precision: {lstm_precision:.4f}")
print(f"Recall:    {lstm_recall:.4f}")
print(f"F1 Score:  {lstm_f1:.4f}")

LSTM Performance:
Precision: 0.6419
Recall:    0.4570
F1 Score:  0.5338


In [65]:
def predict_lstm_sentiment(review):

    lstm_model.eval()

    tokens, length = encode_and_pad(review)

    input_tensor = torch.tensor(
        [tokens],
        dtype=torch.long
    )

    length_tensor = torch.tensor(
        [length],
        dtype=torch.long
    )

    with torch.no_grad():

        output = lstm_model(
            input_tensor,
            length_tensor
        )

        probability = torch.sigmoid(output).item()

    if probability >= 0.5:
        sentiment = "Positive"
    else:
        sentiment = "Negative"

    return sentiment, probability

In [66]:
for review in test_reviews:

    sentiment, probability = predict_lstm_sentiment(review)

    print("\nReview:")
    print(review)

    print("LSTM Prediction:", sentiment)
    print(f"Positive probability: {probability:.4f}")


Review:
This movie was absolutely fantastic. I loved every minute of it.
LSTM Prediction: Positive
Positive probability: 0.8823

Review:
This was one of the worst movies I have ever watched. Completely boring.
LSTM Prediction: Negative
Positive probability: 0.0918

Review:
Amazing acting, wonderful story, and an excellent ending.
LSTM Prediction: Positive
Positive probability: 0.9488

Review:
Terrible movie. Poor acting and a very disappointing story.
LSTM Prediction: Negative
Positive probability: 0.0808


In [67]:
print("===== MODEL COMPARISON =====")

print(f"RNN Test Accuracy : {test_accuracy * 100:.2f}%")
print(f"LSTM Test Accuracy: {lstm_test_accuracy * 100:.2f}%")

print(
    f"\nAccuracy Improvement: "
    f"{(lstm_test_accuracy - test_accuracy) * 100:.2f} percentage points"
)

===== MODEL COMPARISON =====
RNN Test Accuracy : 56.00%
LSTM Test Accuracy: 59.83%

Accuracy Improvement: 3.83 percentage points


## 1. Which model achieved higher test accuracy — RNN or LSTM?

The **LSTM achieved higher test accuracy** than the RNN.

- RNN Test Accuracy: **56.00%**
- LSTM Test Accuracy: **59.83%**
- Improvement: **3.83 percentage points**

Therefore, the **LSTM performed better** on the unseen test dataset.

## 2. Did the LSTM reach a lower training loss than the RNN, or in fewer epochs?

Yes. The LSTM reached a lower training loss than the RNN after 5 epochs.

| Epoch | RNN Loss | LSTM Loss |
|------:|---------:|----------:|
| 1 | 0.6952 | 0.6912 |
| 2 | 0.6742 | 0.6707 |
| 3 | 0.6417 | 0.6307 |
| 4 | 0.5954 | 0.5555 |
| 5 | 0.5241 | **0.4450** |

The final training loss was **0.5241 for the RNN** and **0.4450 for the LSTM**.

Both models were trained for **5 epochs**, so the LSTM did not reach the result in fewer epochs. However, it achieved a lower loss within the same number of epochs.

## 3. Did the RNN and LSTM agree on the predictions for your custom test sentences? If not, which one seems more reliable?

The RNN and LSTM agreed on **3 out of 4** custom review predictions.

| Review | RNN | LSTM |
|---|---|---|
| Absolutely fantastic movie | Positive | Positive |
| Worst movie, completely boring | Negative | Negative |
| Amazing acting and wonderful story | Positive | Positive |
| Terrible movie, poor acting, disappointing story | Positive | **Negative** |

The models disagreed on the fourth review. The LSTM predicted **Negative**, which is more reliable because the review contains strong negative expressions such as *terrible*, *poor acting*, and *disappointing*.

Therefore, based on these custom examples, the **LSTM appears more reliable** than the RNN.

## 4. What is the LSTM doing differently from the RNN?

A standard RNN uses a hidden state to carry information from previous tokens, but it can have difficulty retaining important information over long sequences because of problems such as the **vanishing-gradient problem**.

An LSTM (Long Short-Term Memory) uses a **cell state and three main gates**:

- **Forget gate** – decides what previous information should be discarded.
- **Input gate** – decides what new information should be stored.
- **Output gate** – decides what information should be passed to the next layer.

These gates allow the LSTM to selectively remember or forget information throughout the sequence.

LSTM learn sentiment-related patterns more effectively than the basic RNN. The LSTM achieved a lower final training loss (**0.4450 vs. 0.5241**) and higher test accuracy (**59.83% vs. 56.00%**).

Therefore, the results indicate that the LSTM provided better performance than the simple RNN for this text-classification task.